In [3]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.utils import class_weight
import matplotlib.pyplot as plt
import wandb


In [4]:
# 1. Initialize WandB
wandb.init(
    project="Saka-RHD-Detection",
    config={
        "architecture": "MobileNetV2",
        "dataset": "PhysioNet-CirCOR-Hybrid",
        "learning_rate": 0.0005,
        "epochs": 30,
        "batch_size": 32,
        "img_size": (128, 128)
    }
)
config = wandb.config


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: e-githinji1 (e-githinji1-african-leadership-group) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
# 2. Paths
DATA_PATH = "/content/drive/MyDrive/heart_grades_classified"
MODEL_SAVE_PATH = "saka_v1.h5"

In [9]:
# 3. Load Datasets - Train/Val/Test Split (70/15/15)


# First, get the full dataset to split it manually
full_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_PATH,
    seed=123,
    image_size=config.img_size,
    batch_size=config.batch_size,
    label_mode='int'
)

# Get class names BEFORE any transformations
class_names = full_ds.class_names
print(f"Classes found: {class_names}")

# Calculate dataset sizes
dataset_size = tf.data.experimental.cardinality(full_ds).numpy()
train_size = int(0.7 * dataset_size)
val_size = int(0.15 * dataset_size)
test_size = dataset_size - train_size - val_size

print(f"Total batches: {dataset_size}")
print(f"Train batches: {train_size}")
print(f"Val batches: {val_size}")
print(f"Test batches: {test_size}")

# Split the dataset
train_ds = full_ds.take(train_size)
remaining_ds = full_ds.skip(train_size)
val_ds = remaining_ds.take(val_size)
test_ds = remaining_ds.skip(val_size)

# Optional: Cache and prefetch for performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

# class_names is already defined above, so no need to access from train_ds
print(f"Training with classes: {class_names}")
print(f"Number of classes: {len(class_names)}")

Found 29502 files belonging to 4 classes.
Classes found: ['Grade_0', 'Grade_1', 'Grade_2', 'Grade_3']
Total batches: 922
Train batches: 645
Val batches: 138
Test batches: 139
Training with classes: ['Grade_0', 'Grade_1', 'Grade_2', 'Grade_3']
Number of classes: 4


In [ ]:
# 4. Handle Class Imbalance

# calculate weights so the model pays 20x more attention to Grade 2.
y_train = np.concatenate([y for x, y in train_ds], axis=0)
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights_dict = dict(enumerate(weights))
print(f"Calculated Class Weights: {class_weights_dict}")

# 5. Data Augmentation (Crucial for the minority classes)
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])


In [ ]:
# 4. Build the model
model = tf.keras.Sequential([
    # Add your layers here
    tf.keras.layers.Rescaling(1./255, input_shape=(*config.img_size, 3)),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(len(class_names), activation='softmax')
])

# 5. Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', 'precision', 'auc']
)

# 6. Train the model with validation
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10  # Adjust as needed
)

# 7. Evaluate on test set (ONLY ONCE at the end!)
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

# 8. Get predictions on test set
predictions = model.predict(test_ds)
predicted_classes = np.argmax(predictions, axis=1)

In [ ]:
# 6. Model using MobileNetV2 Transfer Learning

base_model = MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),
    data_augmentation,
    layers.Lambda(lambda x: tf.keras.applications.mobilenet_v2.preprocess_input(x)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(4, activation='softmax')
])

model.compile(
    optimizer=optimizers.Adam(learning_rate=config.learning_rate),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name='top_2_accuracy')]
)

In [ ]:
# 7. Training
print("Training started")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=config.epochs,
    class_weight=class_weights_dict,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(MODEL_SAVE_PATH, save_best_only=True)
    ]
)


In [ ]:
# 8. Evaluation & Final Logging
print(" Training Finished. Evaluating...")
loss, acc, top2 = model.evaluate(val_ds)
wandb.log({"final_val_accuracy": acc, "final_val_loss": loss})

In [ ]:





# 9. Save as TFLite for the Mobile App / Edge
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open("../exported_models/saka_v1.tflite", "wb") as f:
    f.write(tflite_model)

print(f"Model exported to {MODEL_SAVE_PATH} and TFLite version.")
wandb.finish()